# Gaussian Blur Overlay — Video Pipeline
Chops an input video into frames, sends each frame through the HLS Gaussian blur IP, and saves the blurred output.

In [1]:
from pynq import Overlay, allocate
import numpy as np
import cv2
import os
import time

## 1. Configuration

In [2]:
# --- Paths ---
BITSTREAM   = "gaussian.bit"   # .bit file (keep .hwh in same folder)
INPUT_VIDEO = "input.mp4"           # video to process
OUTPUT_DIR  = "output_frames"       # blurred frames saved here

# --- Must match IMG_HEIGHT / IMG_WIDTH in gaussian_blur_hls.h ---
IMG_HEIGHT = 480
IMG_WIDTH  = 640

# --- AXI-Lite register offsets (verify against xgaussian_blur_hw.h) ---
AP_CTRL       = 0x00   # bit 0 = ap_start, bit 1 = ap_done, bit 2 = ap_idle
REG_INPUT_LO  = 0x10   # input pointer  bits [31:0]
REG_INPUT_HI  = 0x14   # input pointer  bits [63:32]
REG_OUTPUT_LO = 0x1c   # output pointer bits [31:0]
REG_OUTPUT_HI = 0x20   # output pointer bits [63:32]

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Overlay

In [3]:
print("Loading overlay...")
ol = Overlay(BITSTREAM)
ip = ol.gaussian_blur_0
print("Overlay loaded.")
ip?

Loading overlay...


Overlay loaded.


## 3. Allocate DMA Buffers


In [4]:
in_buf  = allocate(shape=(IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)
out_buf = allocate(shape=(IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)

print(f"Input  buffer physical address: 0x{in_buf.physical_address:08X}")
print(f"Output buffer physical address: 0x{out_buf.physical_address:08X}")

Input  buffer physical address: 0x78480000
Output buffer physical address: 0x78500000


## 4. Helper — Run One Frame Through the IP

In [5]:
def run_gaussian_blur(gray_frame: np.ndarray) -> np.ndarray:
    """
    Send a single grayscale frame through the HLS Gaussian blur IP.
    gray_frame must be uint8 with shape (IMG_HEIGHT, IMG_WIDTH).
    Returns the blurred frame as a numpy array.
    """
    # Copy frame into physically contiguous input buffer
    in_buf[:] = gray_frame
    in_buf.flush()   # ensure PS cache is written to DDR before PL reads it

    # Write input and output DDR addresses to AXI-Lite control registers
    in_phys  = in_buf.physical_address
    out_phys = out_buf.physical_address

    ip.write(REG_INPUT_LO,  in_phys  & 0xFFFFFFFF)
    ip.write(REG_INPUT_HI,  (in_phys  >> 32) & 0xFFFFFFFF)
    ip.write(REG_OUTPUT_LO, out_phys & 0xFFFFFFFF)
    ip.write(REG_OUTPUT_HI, (out_phys >> 32) & 0xFFFFFFFF)

    # Start the IP (ap_start = bit 0)
    ip.write(AP_CTRL, 0x1)

    # Poll ap_done (bit 1) — for long frames consider an interrupt instead
    while not (ip.read(AP_CTRL) & 0x2):
        pass

    # Invalidate PS cache so we read fresh DDR data written by PL
    out_buf.invalidate()

    return np.array(out_buf)

In [6]:
import cv2

In [7]:
MIN_AREA = 800  # remove small items  
def draw(frame: np.ndarray, image: np.ndarray) -> np.ndarray:
    output = frame.copy()
    image = cv2.Canny(image, 50, 150)
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    cv2.drawContours(output, contours, -1, (0,0,255), 2)
#     found = 0
    
#     for contour in contours:
#         area = cv2.contourArea(contour)
#         if area < MIN_AREA: continue
#         found += 1
 
#         x, y, w, h = cv2.boundingRect(contour)
#         cv2.rectangle(output, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return output

## 5. Process Video

In [8]:
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {total_frames} frames @ {fps:.1f} fps")

frame_idx  = 0
times      = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize to match HLS block dimensions if needed
    if frame.shape[0] != IMG_HEIGHT or frame.shape[1] != IMG_WIDTH:
        frame = cv2.resize(frame, (IMG_WIDTH, IMG_HEIGHT))

    # Convert to grayscale — HLS block is single-channel
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Run through HLS Gaussian blur IP
    t0 = time.perf_counter()
    for i in range(5): gray = run_gaussian_blur(gray)
    blurred = draw(gray, frame)
    times.append(time.perf_counter() - t0)

    # Save blurred frame
    out_path = os.path.join(OUTPUT_DIR, f"frame_{frame_idx:05d}.png")
    cv2.imwrite(out_path, blurred)

    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f"  Processed {frame_idx}/{total_frames} frames")

cap.release()

avg_ms = (sum(times) / len(times)) * 1000 if times else 0
print(f"\nDone. {frame_idx} frames saved to '{OUTPUT_DIR}/'")
print(f"Avg HLS time per frame: {avg_ms:.2f} ms")

Video: 47 frames @ 30.0 fps

Done. 47 frames saved to 'output_frames/'
Avg HLS time per frame: 585.63 ms


## 6. (Optional) Reassemble Frames into a Video

In [9]:
OUTPUT_VIDEO = "output_blurred.avi"

fourcc  = cv2.VideoWriter_fourcc(*'XVID')
writer  = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (IMG_WIDTH, IMG_HEIGHT), isColor=False)

frame_files = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith(".png"))
for fname in frame_files:
    img = cv2.imread(os.path.join(OUTPUT_DIR, fname), cv2.IMREAD_GRAYSCALE)
    writer.write(img)

writer.release()
print(f"Saved reassembled video: {OUTPUT_VIDEO}")

Saved reassembled video: output_blurred.avi


## 7. Free Buffers

In [10]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")

Buffers freed.
